In [ ]:
import metview as mv
from datetime import datetime, timedelta
import os
import pandas as pd
import numpy as np
from multiprocessing import Pool
import sys

#
# PRECIPITATION TYPE VERIFICATION - SAFE PARALLEL VERSION
# --------------------------------------------------------
#
# Uses LIMITED parallelization (2-4 processes only) with optimizations
# Safer than full parallelization but faster than sequential
#
########################################

# ===== CONFIGURATION =====
start_date = datetime(2023, 10, 15)
end_date = datetime(2024, 3, 15)

# Generate dates
fdates = []
current_date = start_date
while current_date <= end_date:
    fdates.append(int(current_date.strftime('%Y%m%d')))
    current_date += timedelta(days=1)

lsteps = list(range(0, 145, 3))
lsteps.extend([150, 156, 162, 168])

# Paths (global for worker processes)
PATH_OROG_in = "/ec/vol/enscal/VERIF_PrecipType/VERIF_ptype_bias/raw/48r1_2023_esuite/"
PATH_MODEL_in_base = "/ec/vol/enscal/VERIF_PrecipType/VERIF_ptype_all/probabilities_ptype/2023_2024/"
PATH_OUTPUT = "/ec/vol/enscal/VERIF_PrecipType/VERIF_ptype_all/DATA_preVerif/2023_2024/"
PATH_SYNOP_in = "/ec/vol/enscal/VERIF_PrecipType/VERIF_ptype_all/SYNOP/2023_2024/"


AREA = [71, -11, 33, 35]
OROG_THRESHOLD = 200

# IMPORTANT: Use ONLY 2-4 processes to avoid Metview conflicts
N_PROCESSES = 3  # Conservative choice for Metview

SYNOP_CODES = {
    'rain': np.array([51, 52, 53, 54, 55, 58, 59, 60, 61, 62, 63, 64, 65, 68, 69, 
                      80, 81, 82, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 99]),
    'snow': np.array([70, 71, 72, 73, 74, 75, 76, 77, 78, 85, 86, 93, 94, 95, 97]),
    'icepellets': np.array([79, 87, 77]),
    'freezrain': np.array([66, 67]),
    'freezdrizzle': np.array([56, 57]),
    'sleet': np.array([68, 69, 83, 84, 87, 88, 89, 90, 93, 94, 95, 97])
}

def classify_synop_vectorized(obs_values, precip_type):
    """Vectorized classification"""
    codes = SYNOP_CODES[precip_type]
    return np.isin(obs_values, codes).astype(int)


def process_single_date(fdate):
    """Process one date - called by worker process"""
    
    PATH_MODEL_in = PATH_MODEL_in_base + str(fdate) + "_0/"
    
    if not os.path.exists(PATH_MODEL_in):
        return (fdate, 0, len(lsteps), 0, "Dir not found")
    
    try:
        # Read orography once
        orog = mv.read(PATH_OROG_in + "orog.grib")
    except Exception as e:
        return (fdate, 0, len(lsteps), 0, f"Orog error: {str(e)[:30]}")
    
    # SYNOP cache
    synop_cache = {}
    all_data = []
    n_success = 0
    n_fail = 0
    
    for step in lsteps:
        
        try:
            # Read model data
            rain_filter = mv.read(PATH_MODEL_in + f"perc_rain_{step}.grib")
            snow_filter = mv.read(PATH_MODEL_in + f"perc_snow_{step}.grib")
            wetsnow_filter = mv.read(PATH_MODEL_in + f"perc_wetsnow_{step}.grib")
            freezrain_filter = mv.read(PATH_MODEL_in + f"perc_freezrain_{step}.grib")
            freezdrizzle_filter = mv.read(PATH_MODEL_in + f"perc_freezdrizzle_{step}.grib")
            icepellets_filter = mv.read(PATH_MODEL_in + f"perc_icepellets_{step}.grib")
            sleet_filter = mv.read(PATH_MODEL_in + f"perc_sleet_{step}.grib")
        except:
            n_fail += 1
            continue
        
        fdate_SYN = int(mv.yyyymmdd(mv.valid_date(rain_filter)))
        lstep = step % 24
        
        # Check SYNOP cache
        if fdate_SYN not in synop_cache:
            try:
                synop_cache[fdate_SYN] = mv.read(PATH_SYNOP_in + f"SYNOP_mannual_{fdate_SYN}.bufr")
            except:
                n_fail += 1
                continue
        
        SYNOP_data = synop_cache[fdate_SYN]
        
        obs_max = mv.obsfilter(output="geopoints", parameter=20003, time=lstep,
                               location_filter="area", area=AREA, data=SYNOP_data)
        observ_height = mv.obsfilter(output="geopoints", parameter=7001, time=lstep,
                                      location_filter="area", area=AREA, data=SYNOP_data)
        
        obs_height = mv.subsample(observ_height, obs_max)
        
        try:
            n_weather = len(mv.values(obs_max))
            n_height = len(mv.values(obs_height))
            if n_weather != n_height:
                n_fail += 1
                continue
        except:
            n_fail += 1
            continue
        
        Locations_orog = mv.nearest_gridpoint(orog, obs_max)
        diff_orog = obs_height - Locations_orog
        diff_orog_ok = (diff_orog < OROG_THRESHOLD) & (diff_orog > -OROG_THRESHOLD)
        
        obs_ = mv.filter(obs_max, diff_orog_ok == 1)
        obs_height_ = mv.filter(obs_height, diff_orog_ok == 1)
        
        Miss = (obs_ == 508) | (obs_ == 509) | (obs_ == 510) | (obs_ == 511) | (obs_ > 100)
        obs = mv.filter(obs_, Miss == 0)
        obs_height_filtered = mv.filter(obs_height_, Miss == 0)
        
        try:
            obs_values = mv.values(obs)
            n_obs = len(obs_values)
            if n_obs == 0:
                n_fail += 1
                continue
            
            obs_lats = mv.latitudes(obs)
            obs_lons = mv.longitudes(obs)
            obs_elevs = mv.values(obs_height_filtered)
        except:
            n_fail += 1
            continue
        
        prob_rain = mv.values(mv.nearest_gridpoint(rain_filter, obs))
        prob_snow = mv.values(mv.nearest_gridpoint(snow_filter, obs))
        prob_wetsnow = mv.values(mv.nearest_gridpoint(wetsnow_filter, obs))
        prob_sleet = mv.values(mv.nearest_gridpoint(sleet_filter, obs))
        prob_freezrain = mv.values(mv.nearest_gridpoint(freezrain_filter, obs))
        prob_freezdrizzle = mv.values(mv.nearest_gridpoint(freezdrizzle_filter, obs))
        prob_icepellets = mv.values(mv.nearest_gridpoint(icepellets_filter, obs))
        
        # Vectorized classification
        obs_array = np.array(obs_values)
        obs_rain = classify_synop_vectorized(obs_array, 'rain')
        obs_snow = classify_synop_vectorized(obs_array, 'snow')
        obs_icepellets = classify_synop_vectorized(obs_array, 'icepellets')
        obs_freezrain = classify_synop_vectorized(obs_array, 'freezrain')
        obs_freezdrizzle = classify_synop_vectorized(obs_array, 'freezdrizzle')
        obs_sleet = classify_synop_vectorized(obs_array, 'sleet')
        
        step_data = pd.DataFrame({
            'forecast_date': fdate, 'valid_date': fdate_SYN, 'forecast_step': step, 'valid_hour': lstep,
            'latitude': obs_lats, 'longitude': obs_lons, 'elevation': obs_elevs, 'synop_code': obs_values,
            'obs_rain': obs_rain, 'obs_snow': obs_snow, 'obs_wetsnow': 0, 'obs_sleet': obs_sleet,
            'obs_freezrain': obs_freezrain, 'obs_freezdrizzle': obs_freezdrizzle, 'obs_icepellets': obs_icepellets,
            'prob_rain': prob_rain, 'prob_snow': prob_snow, 'prob_wetsnow': prob_wetsnow, 'prob_sleet': prob_sleet,
            'prob_freezrain': prob_freezrain, 'prob_freezdrizzle': prob_freezdrizzle, 'prob_icepellets': prob_icepellets,
        })
        
        all_data.append(step_data)
        n_success += 1
    
    # Save
    if len(all_data) > 0:
        date_df = pd.concat(all_data, ignore_index=True)
        output_file = PATH_OUTPUT + f"matched_pairs_{fdate}.parquet"
        date_df.to_parquet(output_file, compression='snappy', index=False)
        n_records = len(date_df)
        return (fdate, n_success, n_fail, n_records, "OK")
    else:
        return (fdate, n_success, n_fail, 0, "No data")


if __name__ == '__main__':
    
    # Create output directory
    if not os.path.exists(PATH_OUTPUT):
        os.makedirs(PATH_OUTPUT, exist_ok=True)
    
    print("="*70)
    print("PRECIPITATION TYPE VERIFICATION - SAFE PARALLEL")
    print("="*70)
    print(f"Dates: {len(fdates)} ({start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')})")
    print(f"Steps per date: {len(lsteps)}")
    print(f"Parallel processes: {N_PROCESSES} (limited for Metview stability)")
    print(f"Output: {PATH_OUTPUT}")
    print("="*70)
    print()
    
    total_start = datetime.now()
    
    print(f"Starting parallel processing with {N_PROCESSES} workers...")
    print("(This may take 30-60 minutes for 90 dates)")
    print()
    
    # Process in parallel with LIMITED processes
    try:
        with Pool(processes=N_PROCESSES) as pool:
            results = pool.map(process_single_date, fdates)
    except KeyboardInterrupt:
        print("\n\nInterrupted by user!")
        sys.exit(1)
    except Exception as e:
        print(f"\n\nERROR in parallel processing: {e}")
        sys.exit(1)
    
    total_elapsed = (datetime.now() - total_start).total_seconds()
    
    # Summary
    print("\n" + "="*70)
    print("PROCESSING COMPLETE!")
    print("="*70)
    print(f"Total time: {total_elapsed/60:.1f} minutes ({total_elapsed/3600:.1f} hours)")
    print()
    print("Results:")
    print(f"{'Date':<12} {'Success':<10} {'Failed':<10} {'Records':<12} {'Status':<20}")
    print("-"*70)
    
    total_records = 0
    total_success = 0
    for fdate, n_succ, n_fail, n_rec, status in results:
        total_records += n_rec
        total_success += (n_rec > 0)
        print(f"{fdate:<12} {n_succ:<10} {n_fail:<10} {n_rec:<12,} {status:<20}")
    
    print("-"*70)
    print(f"{'TOTAL':<12} {'':<10} {'':<10} {total_records:<12,}")
    print()
    print(f"Successful dates: {total_success}/{len(fdates)}")
    print(f"Average time per date: {total_elapsed/len(fdates):.1f} seconds")
    if total_records > 0:
        print(f"Average records per date: {total_records/len(fdates):,.0f}")
    
    print(f"\nSpeedup vs sequential: ~{N_PROCESSES}× faster")

PRECIPITATION TYPE VERIFICATION - SAFE PARALLEL
Dates: 153 (2023-10-15 to 2024-03-15)
Steps per date: 53
Parallel processes: 3 (limited for Metview stability)
Output: /ec/vol/enscal/VERIF_PrecipType/VERIF_ptype_all/DATA_preVerif/2023_2024/

Starting parallel processing with 3 workers...
(This may take 30-60 minutes for 90 dates)

Python - ERROR  - 20260204.141249 - Line 0 in 'PythonScript': Cannot read /ec/vol/enscal/VERIF_PrecipType/VERIF_ptype_all/probabilities_ptype/2023_2024/20231110_0/perc_rain_0.grib. Type BAD is not supported.
Python - ERROR  - 20260204.141249 - Line 0 in 'PythonScript': Cannot read /ec/vol/enscal/VERIF_PrecipType/VERIF_ptype_all/probabilities_ptype/2023_2024/20231110_0/perc_rain_3.grib. Type BAD is not supported.
Python - ERROR  - 20260204.141249 - Line 0 in 'PythonScript': Cannot read /ec/vol/enscal/VERIF_PrecipType/VERIF_ptype_all/probabilities_ptype/2023_2024/20231110_0/perc_rain_6.grib. Type BAD is not supported.
Python - ERROR  - 20260204.141249 - Line 0 i